# Preprocessing: MORE Solar Park (Project 04)
الملفات المطلوبة جنب الـ notebook: `Weather_Data.csv` و`Plant_Active_Power.csv` و`Array_Power.csv`.
الـ notebook ده بيعمل preprocessing بس: مفيش modeling ولا feature engineering ولا حقن anomalies.
المبدأ الأساسي: أخطاء الحساسات بتتحول لـ NaN مع حفظ القيمة الأصلية، أما الأحداث التشغيلية الحقيقية (أعطال الـ arrays وتوقف المحطة) فبتفضل في الداتا وبتتعلّم كـ evidence للـ anomaly detection.
المخرجات: ملفات train/val/test مقسّمة حسب الغرض، وملفات `splits.json` و`scaler_params.json` و`events.csv` و`data_quality_report.json`.

In [1]:
import gzip
import hashlib
import json
import os
import platform
import random
import sys

import numpy as np
import pandas as pd

random.seed(42)
np.random.seed(42)

DATA_DIR = "."      # غيّر المسار لو الملفات في مكان تاني
OUT_DIR = "."
REPORT = {}

# ---- الإعدادات (كل العتبات في مكان واحد) ----
LOCAL_TZ = "Europe/Athens"      # الداتا بتوقيت محلي فيه DST (اتأكدنا من ده تحت)
STEP_MIN = 15
SITE_LAT, SITE_LON = 40.5, 21.8     # تقدير تقريبي (خط الطول من وقت الظهر الشمسي)، والـ night mask مش حساس لخط العرض
LABEL_SHIFT_MIN = 7.5               # الـ timestamp هو نهاية الفترة، فنحسب الشمس عند منتصفها
NIGHT_ELEV, DAY_ELEV = -6.0, 10.0   # ارتفاع الشمس (بالدرجات) لتعريف الليل والنهار
WSPD_MAX, TEMP_MIN, IRR_MAX = 40.0, -30.0, 1400.0
STUCK_MIN_ROWS = 12                 # قراءة ثابتة 3 ساعات فأكتر = حساس واقف
TEMP_RECOVERY_MIN_C = -2.0          # نهاية فترة تعافي حساس الحرارة: أول يوم أقل حرارة فيه فوق -2
IRR_FAULT_NIGHT_MEDIAN = 8.0        # وسيط إشعاع الليل اليومي فوق 8 W/m2 = حساس الإشعاع عطلان
OFFSET_WINDOW_DAYS, OFFSET_MIN_DAYS = 7, 3
FFILL_LIMIT, IRR_FFILL_LIMIT = 4, 2 # أقصى عدد صفوف forward-fill (ساعة، و30 دقيقة للإشعاع)
ZERO_IRR, ZERO_PU = 100.0, 0.02     # إشعاع عالي وقدرة تقريباً صفر = توقف محتمل للمحطة
ARR_MEDIAN_MIN, ARR_MIN_RUN = 0.3, 8
PURGE_ROWS = 192                    # 48 ساعة
WARMUP_ROWS = 1344                  # 14 يوم (للـ lags والـ rolling)


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def runs(mask, min_len=1):
    """(start, end, rows) لكل فترة متصلة True في Series بوليانية."""
    m = mask.fillna(False).astype(bool)
    grp = (m != m.shift()).cumsum()
    out = []
    for _, s in m[m].groupby(grp[m]):
        if len(s) >= min_len:
            out.append((s.index[0], s.index[-1], len(s)))
    return out


def identical_runs(s, min_rows):
    grp = (s != s.shift()).cumsum()
    return (s.groupby(grp).transform("size") >= min_rows) & s.notna()


def solar_elevation(t_utc, lat, lon):
    """ارتفاع الشمس بالدرجات (معادلات NOAA المبسطة)."""
    n = (t_utc - pd.Timestamp("2000-01-01 12:00", tz="UTC")) / pd.Timedelta(days=1)
    n = np.asarray(n, dtype=float)
    L = np.deg2rad((280.460 + 0.9856474 * n) % 360)
    g = np.deg2rad((357.528 + 0.9856003 * n) % 360)
    lam = L + np.deg2rad(1.915) * np.sin(g) + np.deg2rad(0.020) * np.sin(2 * g)
    eps = np.deg2rad(23.439 - 0.0000004 * n)
    dec = np.arcsin(np.sin(eps) * np.sin(lam))
    ra = np.arctan2(np.cos(eps) * np.sin(lam), np.cos(lam))
    lst = np.deg2rad(((280.46061837 + 360.98564736629 * n) + lon) % 360)
    ha = lst - ra
    la = np.deg2rad(lat)
    return np.rad2deg(np.arcsin(np.sin(la) * np.sin(dec) + np.cos(la) * np.cos(dec) * np.cos(ha)))


def dump_json(obj, path):
    def default(o):
        if isinstance(o, np.integer): return int(o)
        if isinstance(o, np.floating): return None if np.isnan(o) else float(o)
        if isinstance(o, np.bool_): return bool(o)
        return str(o)
    with open(path, "w") as fh:
        json.dump(obj, fh, indent=2, sort_keys=True, default=default)


REPORT["environment"] = {"python": sys.version.split()[0], "pandas": pd.__version__, "numpy": np.__version__,
                         "platform": platform.platform(), "seed": 42}
FILES = ("Weather_Data.csv", "Plant_Active_Power.csv", "Array_Power.csv")
REPORT["input_sha256"] = {f: sha256_file(f"{DATA_DIR}/{f}") for f in FILES}
print(REPORT["environment"])

{'python': '3.12.3', 'pandas': '3.0.2', 'numpy': '2.4.4', 'platform': 'Linux-6.18.44-fc-v153-x86_64-with-glibc2.39', 'seed': 42}


## 1) تحميل الملفات
ملف الطقس فيه رمز درجة الحرارة بترميز latin-1، فبنقرأ الكل بيه.
الأعمدة الأصلية طويلة، فبنعيد تسميتها باختصارات ثابتة.

In [2]:
W = pd.read_csv(f"{DATA_DIR}/Weather_Data.csv", encoding="latin-1")
P = pd.read_csv(f"{DATA_DIR}/Plant_Active_Power.csv", encoding="latin-1")
A = pd.read_csv(f"{DATA_DIR}/Array_Power.csv", encoding="latin-1")
print("shapes:", W.shape, P.shape, A.shape)
assert (W["Timestamp"] == P["Timestamp"]).all() and (W["Timestamp"] == A["Timestamp"]).all(), "timestamps differ between files"

wcols = ["temp_c", "irr_wm2", "mod_temp_c", "precip_mmh", "rh_pct", "wdir_deg", "wspd_ms"]
arr_cols = [f"arr{i}" for i in range(1, 15)]
assert W.shape[1] == 8 and A.shape[1] == 15 and P.shape[1] == 2
REPORT["original_headers"] = {"weather": list(W.columns), "plant": list(P.columns), "arrays": list(A.columns)}
naive = pd.to_datetime(W["Timestamp"], format="%d/%m/%y %H:%M")     # day-first (مثال: 13/1/16)
W.columns = ["Timestamp"] + wcols
P.columns = ["Timestamp", "plant_pu"]
A.columns = ["Timestamp"] + arr_cols

shapes: (280512, 8) (280512, 2) (280512, 15)


## 2) الـ timestamps والتوقيت الصيفي (أهم خطوة)
التوقيت في الملفات محلي وفيه DST، فبنلاقي 32 قراءة مكررة (الساعة المعادة في الخريف) و8 قفزات للأمام (الساعة المفقودة في الربيع).
عدد الصفوف بيساوي بالظبط عدد خطوات الـ 15 دقيقة في 8 سنين، يعني كل صف خطوة حقيقية واحدة.
عشان كده بنعيد بناء الـ UTC من ترتيب الصفوف، وبنتأكد بطريقتين مستقلتين إن النتيجة صح.

In [3]:
d_naive = naive.diff().dropna()
n_dup = int(naive.duplicated().sum())
n_spring = int((d_naive == pd.Timedelta("75min")).sum())     # الساعة المفقودة
n_fall = int((d_naive == pd.Timedelta("-45min")).sum())      # الساعة المعادة
print("duplicated naive stamps:", n_dup, "| spring jumps:", n_spring, "| fall repeats:", n_fall)

first_utc = naive.iloc[0].tz_localize(LOCAL_TZ).tz_convert("UTC")
idx = pd.date_range(first_utc, periods=len(W), freq=f"{STEP_MIN}min", name="time_utc")

# تحقق 1: الوقت المحلي المعاد بناؤه بيطابق الأصلي في كل الصفوف
match = float((idx.tz_convert(LOCAL_TZ).tz_localize(None) == pd.DatetimeIndex(naive)).mean())
# تحقق 2 (مستقل): آخر صف محسوب من نهاية الملف بيطابق آخر صف في الـ index
last_expected = naive.iloc[-1].tz_localize(LOCAL_TZ).tz_convert("UTC")
print(f"rebuilt local == original in {match:.4%} of rows | last row matches: {idx[-1] == last_expected}")
assert match == 1.0 and idx[-1] == last_expected
assert n_spring == n_fall and n_dup == 4 * n_fall

REPORT["time_reconstruction"] = {
    "method": "UTC rebuilt from row order (each row = +15 min); validated against the original wall-clock stamps",
    "local_tz": LOCAL_TZ, "first_utc": str(idx[0]), "last_utc": str(idx[-1]), "rows": len(idx),
    "duplicated_naive_stamps": n_dup, "spring_forward_jumps": n_spring, "fall_back_repeats": n_fall,
    "local_match_ratio": match, "stamp_semantics": "interval END (weak evidence, effect is only 7.5 min and does not matter for the day/night masks)",
}

for df_ in (W, P, A):
    df_.index = idx
raw = W[wcols].join(P[["plant_pu"]]).join(A[arr_cols])
assert raw.index.to_series().diff().dropna().eq(pd.Timedelta(f"{STEP_MIN}min")).all()
print(raw.shape, raw.index.min(), "->", raw.index.max())

duplicated naive stamps: 32 | spring jumps: 8 | fall repeats: 8
rebuilt local == original in 100.0000% of rows | last row matches: True
(280512, 22) 2015-12-31 22:15:00+00:00 -> 2023-12-31 22:00:00+00:00


## 3) هندسة الشمس وتعريف الليل
تعريف الليل بساعة الساعة (مثلاً 22 لـ 4) غلط لأنه بيتغير مع الفصول والـ DST.
بنحسب ارتفاع الشمس عند منتصف كل فترة، والليل = ارتفاع الشمس أقل من -6 درجات.
الجدول تحت بيختبر 3 حاجات: الإزاحة الزمنية الكبيرة (±60 دقيقة) بتأكد إن التوقيت المحلي هو EET/EEST مش CET، والإزاحات الصغيرة بتشير بفارق بسيط إن الـ timestamp نهاية الفترة، وخط العرض بين 38 و42 مش بيأثر تقريباً.
تحقق مطابقة الوقت المحلي في الخطوة اللي فاتت بيثبت قاعدة الـ DST بس، لأن كل دول الاتحاد الأوروبي بتبدّل في نفس اللحظة، فالتحقق من المنطقة الزمنية نفسها بييجي من هندسة الشمس.

In [4]:
mid = idx - pd.Timedelta(minutes=LABEL_SHIFT_MIN)
elev = pd.Series(solar_elevation(mid, SITE_LAT, SITE_LON), index=idx, name="solar_elev_deg")

# فترة سليمة (قبل أي عطل في حساس الإشعاع) لاختبار الفرضيات
healthy = raw.loc["2016-01-01":"2021-11-30", "irr_wm2"].dropna()
rows = []
for shift in (-60.0, -15.0, -7.5, 0.0, 7.5, 15.0, 60.0):
    for lat in (38.0, 40.5, 42.0):
        e = solar_elevation(healthy.index + pd.Timedelta(minutes=shift), lat, SITE_LON)
        viol = int(((e < NIGHT_ELEV) & (healthy.values > 2)).sum() + ((e > 4) & (healthy.values <= 0.5)).sum())
        rows.append({"label_shift_min": shift, "lat": lat, "violations": viol})
ev = pd.DataFrame(rows).pivot(index="label_shift_min", columns="lat", values="violations")
print("عدد المخالفات (إشعاع بالليل أو صفر بالنهار): الأقل = الأصح")
print(ev)
REPORT["geometry_evidence_violations"] = {str(k): v for k, v in ev.to_dict().items()}
best_small = ev.loc[[-15.0, -7.5, 0.0, 7.5, 15.0]].min().min()
assert ev.loc[[-60.0, 60.0]].min().min() > 3 * best_small, "time-zone offset is not clearly identified by solar geometry"

عدد المخالفات (إشعاع بالليل أو صفر بالنهار): الأقل = الأصح
lat              38.0  40.5  42.0
label_shift_min                  
-60.0            6997  6690  6531
-15.0             364   371   392
-7.5              324   319   329
 0.0              336   329   376
 7.5              401   460   485
 15.0             619   651   849
 60.0            9973  9595  9397


## 4) تدقيق الداتا الخام (قبل أي تعديل)

In [5]:
audit = pd.DataFrame({
    "nulls": raw[wcols + ["plant_pu"]].isna().sum(),
    "null_%": (raw[wcols + ["plant_pu"]].isna().mean() * 100).round(2),
    "min": raw[wcols + ["plant_pu"]].min().round(3),
    "max": raw[wcols + ["plant_pu"]].max().round(3),
})
print(audit)
print("array columns null:", int(raw[arr_cols].isna().sum().sum()), "| plant range:", raw["plant_pu"].min(), raw["plant_pu"].max())
print("rows with ALL weather null:", int(raw[wcols].isna().all(axis=1).sum()))
REPORT["raw_audit"] = audit.reset_index().rename(columns={"index": "column"}).to_dict(orient="records")

            nulls  null_%     min       max
temp_c       4844    1.73 -39.118    41.479
irr_wm2      4869    1.74   0.000  1177.137
mod_temp_c   4796    1.71 -13.392    69.998
precip_mmh   4990    1.78   0.000    13.025
rh_pct       5468    1.95   8.999    99.860
wdir_deg     5452    1.94  26.105   401.598
wspd_ms      5447    1.94   0.000    89.035
plant_pu        0    0.00   0.000     0.918
array columns null: 0 | plant range: 0.0 0.918051842
rows with ALL weather null: 4794


## 5) تنضيف الطقس
القواعد:
1. **سرعة الرياح فوق 40 م/ث** (قراءتين فيهم 64 و89) خطأ حساس، فتتحول NaN.
2. **حساس حرارة الهواء في مارس 2022:** قراءات حوالي -39°C وتعافي تدريجي، فبنعتبر الفترة كلها NaN لحد ما أقل حرارة يومية ترجع فوق -2°C.
3. **حرارة الـ module الثابتة** (12 صف فأكتر بنفس القيمة، ثابتة على 70°C) تتحول NaN.
4. **اتجاه الرياح فوق 360** مجرد wrap، فبنعمل `% 360` ومش بنعتبره خطأ.
5. **حساس الإشعاع:** لو وسيط إشعاع الليل في يوم فوق 8 W/m2 فالحساس عطلان في اليوم ده كله، فالإشعاع يتحول NaN.
6. **إزاحة ثابتة في الحساس** (حوالي 5 W/m2 بالليل) بنطرحها بتقدير causal (وسيط آخر 7 أيام)، مش بنمسح البيانات.
القيم الأصلية لأي قيمة اتغيّرت بتتحفظ في ملف الـ flags.

In [6]:
w = raw[wcols].copy()
raw_nan = w.isna()
rule_nan = pd.DataFrame(False, index=idx, columns=wcols)
orig_vals = {c: pd.Series(np.nan, index=idx) for c in ("temp_c", "irr_wm2", "mod_temp_c", "wspd_ms")}
rule_hits, rule_rows = {}, {}


def set_nan(col, mask, rule):
    m = mask.fillna(False) & w[col].notna()
    if col in orig_vals:
        orig_vals[col] = orig_vals[col].where(~m, w[col])
    w.loc[m, col] = np.nan
    rule_nan.loc[m, col] = True
    rule_hits[rule] = rule_hits.get(rule, 0) + int(m.sum())
    rule_rows[rule] = m if rule not in rule_rows else (rule_rows[rule] | m)
    return m


# 1) سرعة الرياح
set_nan("wspd_ms", w["wspd_ms"] > WSPD_MAX, "wspd_gt_40")

# 2) حساس حرارة الهواء: فترة العطل + التعافي
bad_temp = w["temp_c"] < TEMP_MIN
ep_start = w.index[bad_temp].min()
dmin = w["temp_c"].resample("D").min()
back = dmin[(dmin.index > ep_start.normalize()) & (dmin > TEMP_RECOVERY_MIN_C)]
ep_end = back.index[0]
episode = pd.Series((w.index >= ep_start) & (w.index < ep_end), index=idx)
assert episode[bad_temp].all(), "temp < -30 rows outside the failure episode"
set_nan("temp_c", episode, "temp_sensor_episode")

# 3) حرارة الـ module الثابتة
stuck = identical_runs(w["mod_temp_c"], STUCK_MIN_ROWS)
set_nan("mod_temp_c", stuck, "modt_stuck")

# 4) اتجاه الرياح
wrapped = w["wdir_deg"] > 360
w["wdir_deg"] = w["wdir_deg"] % 360
rule_hits["wdir_wrapped_mod360"] = int(wrapped.sum())

# 5) أعطال حساس الإشعاع (وسيط الإشعاع الليلي اليومي)
is_night = elev < NIGHT_ELEV
is_day = elev > DAY_ELEV
night_irr = w["irr_wm2"].where(is_night)
day_key = idx.normalize()
all_days = pd.date_range(idx[0].normalize(), idx[-1].normalize(), freq="D")
dn = night_irr.groupby(day_key).median().where(night_irr.groupby(day_key).count() >= 8).reindex(all_days)
fault_days = dn.index[dn > IRR_FAULT_NIGHT_MEDIAN]
irr_fault = pd.Series(day_key.isin(fault_days), index=idx)
set_nan("irr_wm2", irr_fault, "irr_fault_window")
fault_windows = [(s.date(), e.date()) for s, e, _ in runs(pd.Series(dn > IRR_FAULT_NIGHT_MEDIAN, index=all_days))]

# 6) إزاحة الإشعاع الثابتة (causal: وسيط الأيام السبعة اللي فاتت بس)
dn_ok = dn.where(dn <= IRR_FAULT_NIGHT_MEDIAN)
offset_day = (dn_ok.shift(1).rolling(OFFSET_WINDOW_DAYS, min_periods=OFFSET_MIN_DAYS).median()
              .fillna(0.0).clip(0.0, IRR_FAULT_NIGHT_MEDIAN))
offset_row = pd.Series(offset_day.reindex(day_key).to_numpy(), index=idx, name="irr_offset_applied_wm2")
w["irr_wm2"] = (w["irr_wm2"] - offset_row).clip(lower=0.0)
off_rows = (offset_row >= 1.0) & ~irr_fault

# 7) حدود فيزيائية إضافية (متوقع صفر مخالفات)
set_nan("irr_wm2", w["irr_wm2"] > IRR_MAX, "irr_gt_1400")

resid = w["irr_wm2"][is_night & ~irr_fault].dropna()
resid_by_year = resid.groupby(resid.index.year).median().round(2).to_dict()
print("median night irradiance AFTER offset correction, by year:", resid_by_year)
assert max(resid_by_year.values()) <= 0.5, "night offset not removed"
REPORT["night_irradiance_median_after_correction"] = {str(k): v for k, v in resid_by_year.items()}

print("episode (temp sensor):", ep_start, "->", ep_end)
print("irradiance fault windows (UTC days):", fault_windows)
print("offset period rows:", int(off_rows.sum()), "|", off_rows[off_rows].index.min(), "->", off_rows[off_rows].index.max())
print("rule hits:", rule_hits)

median night irradiance AFTER offset correction, by year: {2015: 0.0, 2016: 0.0, 2017: 0.0, 2018: 0.0, 2019: 0.0, 2020: 0.0, 2021: 0.0, 2022: 0.0, 2023: 0.0}
episode (temp sensor): 2022-03-22 14:30:00+00:00 -> 2022-03-26 00:00:00+00:00
irradiance fault windows (UTC days): [(datetime.date(2020, 5, 6), datetime.date(2020, 5, 6)), (datetime.date(2021, 12, 4), datetime.date(2022, 2, 22)), (datetime.date(2023, 10, 12), datetime.date(2023, 12, 31))]
offset period rows: 40032 | 2022-08-21 00:00:00+00:00 -> 2023-10-11 23:45:00+00:00
rule hits: {'wspd_gt_40': 2, 'temp_sensor_episode': 326, 'modt_stuck': 60, 'wdir_wrapped_mod360': 1597, 'irr_fault_window': 15591, 'irr_gt_1400': 0}


### القيم الناقصة الحقيقية: فجوات الطقس
الفجوات القصيرة (حتى ساعة للقيم العادية و30 دقيقة للإشعاع) بتتعوض بـ forward-fill محدود، وبس للقراءات الناقصة فعلاً (مش اللي اتحولت NaN بقاعدة).
أي فجوة أطول بتفضل NaN ومتعلّمة، ومنخترعش طقس.

In [7]:
fill_ok = raw_nan & ~rule_nan
imputed = pd.DataFrame(False, index=idx, columns=wcols)
for c in wcols:
    lim = IRR_FFILL_LIMIT if c == "irr_wm2" else FFILL_LIMIT
    f = w[c].ffill(limit=lim)
    m = fill_ok[c] & w[c].isna() & f.notna()
    w.loc[m, c] = f[m]
    imputed[c] = m

rad = np.deg2rad(w["wdir_deg"])
w["wind_u"] = -w["wspd_ms"] * np.sin(rad)
w["wind_v"] = -w["wspd_ms"] * np.cos(rad)
weather_cols = wcols + ["wind_u", "wind_v"]

wx_imputed_count = imputed.sum(axis=1).astype(int)
wx_unfilled_any = w[wcols].isna().any(axis=1)
all_missing_raw = raw_nan.all(axis=1)
print("imputed values:", int(imputed.sum().sum()), "| rows still with any weather NaN:", int(wx_unfilled_any.sum()),
      "| rows with all weather missing (raw):", int(all_missing_raw.sum()))

# فحص الاتساق بعد التنضيف: مفيش فرق حرارة (module - ambient) مستحيل بالنهار
chk = (w["mod_temp_c"] - w["temp_c"])[is_day & (w["irr_wm2"] > 300)].dropna()
print("daytime module-ambient diff: max =", round(float(chk.max()), 1))
assert chk.max() <= 45

imputed values: 921 | rows still with any weather NaN: 21109 | rows with all weather missing (raw): 4794
daytime module-ambient diff: max = 37.6


## 6) أحداث تشغيلية حقيقية (evidence للـ anomaly detection)
دي **مش** أخطاء داتا، فمبتتحذفش ومبتتعدّلش.
- **توقف محتمل للمحطة:** إشعاع أكبر من 100 W/m2 والشمس فوق 10 درجات والقدرة أقل من 0.02 p.u.
- **عطل array:** قيمة الـ array صفر والـ median بتاع الـ arrays فوق 0.3 والشمس فوق 10 درجات.
لاحظ إن أي قاعدة بتعتمد على إشعاع قليل بدون شرط ارتفاع الشمس بتطلّع آلاف الإنذارات الكاذبة (القدرة المتوقعة عند 50 W/m2 حوالي 0.04 p.u. بس).

In [8]:
plant = raw["plant_pu"]
irr = w["irr_wm2"]
cand_zero = is_day & (irr > ZERO_IRR) & (plant < ZERO_PU)

arr_med = raw[arr_cols].median(axis=1)
arr_out = pd.DataFrame({c: (raw[c] == 0) & (arr_med > ARR_MEDIAN_MIN) & is_day for c in arr_cols})
arr_out_count = arr_out.sum(axis=1).astype(int)

# مقارنة بقاعدة ساعة-اليوم الشائعة: عدد الإنذارات الكاذبة المحتملة
naive_rule = (raw["irr_wm2"] > 50) & (plant < 0.05)
print("plant-zero candidates (daytime, irr>100):", int(cand_zero.sum()), "in", len(runs(cand_zero)), "events")
print("naive rule (irr>50 & plant<0.05, no sun-elevation filter):", int(naive_rule.sum()), "rows")
print("array outage rows per array:", arr_out.sum().to_dict())

events = []
def add_event(kind, start, end, rows, detail, action):
    events.append({"type": kind, "start_utc": start, "end_utc": end, "rows": int(rows), "detail": detail, "action": action})

for s, e, n in runs(all_missing_raw, min_len=4):
    add_event("weather_all_missing", s, e, n, "all 7 weather sensors missing", "kept NaN (gap > fill limit)" if n > FFILL_LIMIT else "filled")
part = raw_nan.any(axis=1) & ~all_missing_raw
for s, e, n in runs(part, min_len=24):
    cols_ = raw_nan.loc[s:e].mean()
    add_event("weather_partial_missing", s, e, n, "missing: " + ",".join(cols_[cols_ > 0.5].index), "kept NaN (gap > fill limit)")
for s, e, n in runs(irr_fault):
    add_event("irradiance_sensor_fault", s, e, n, "night-median irradiance above %.0f W/m2" % IRR_FAULT_NIGHT_MEDIAN, "irradiance set to NaN; original in flags")
for s, e, n in runs(off_rows):
    add_event("irradiance_offset", s, e, n, "constant night offset ~%.1f W/m2" % float(offset_row[off_rows].median()), "causal offset subtracted")
add_event("temp_sensor_episode", ep_start, ep_end - pd.Timedelta(minutes=STEP_MIN), int(episode.sum()), "ambient temp ~ -39C then recovery", "temp set to NaN; original in flags")
for s, e, n in runs(stuck):
    add_event("module_temp_stuck", s, e, n, "constant module temperature", "set to NaN; original in flags")
for s, e, n in runs(cand_zero):
    allz = int((raw.loc[s:e, arr_cols] == 0).all(axis=1).sum())
    add_event("plant_zero_daytime", s, e, n, f"rows with all arrays zero: {allz}", "kept unchanged (candidate anomaly)")
for c in arr_cols:
    for s, e, n in runs(arr_out[c], min_len=ARR_MIN_RUN):
        add_event("array_outage", s, e, n, c, "kept unchanged (candidate anomaly)")

events_df = pd.DataFrame(events).sort_values("start_utc").reset_index(drop=True)
print(events_df["type"].value_counts().to_dict())
events_df.head(8)

plant-zero candidates (daytime, irr>100): 322 in 49 events
naive rule (irr>50 & plant<0.05, no sun-elevation filter): 10180 rows
array outage rows per array: {'arr1': 1337, 'arr2': 147, 'arr3': 2683, 'arr4': 670, 'arr5': 382, 'arr6': 255, 'arr7': 356, 'arr8': 309, 'arr9': 338, 'arr10': 269, 'arr11': 279, 'arr12': 137, 'arr13': 171, 'arr14': 125}


{'array_outage': 271, 'plant_zero_daytime': 49, 'weather_all_missing': 24, 'weather_partial_missing': 7, 'irradiance_sensor_fault': 3, 'module_temp_stuck': 1, 'temp_sensor_episode': 1, 'irradiance_offset': 1}


,type,start_utc,end_utc,rows,detail,action
0,plant_zero_daytime,2016-01-14 13:30:00+00:00,2016-01-14 13:30:00+00:00,1,rows with all arrays zero: 0,kept unchanged (candidate anomaly)
1,plant_zero_daytime,2016-03-08 15:30:00+00:00,2016-03-08 15:30:00+00:00,1,rows with all arrays zero: 0,kept unchanged (candidate anomaly)
2,plant_zero_daytime,2016-03-23 08:00:00+00:00,2016-03-23 08:15:00+00:00,2,rows with all arrays zero: 2,kept unchanged (candidate anomaly)
3,weather_all_missing,2016-03-23 08:00:00+00:00,2016-03-23 11:45:00+00:00,16,all 7 weather sensors missing,kept NaN (gap > fill limit)
4,weather_all_missing,2016-04-07 19:15:00+00:00,2016-04-09 17:15:00+00:00,185,all 7 weather sensors missing,kept NaN (gap > fill limit)
5,weather_all_missing,2016-04-27 04:30:00+00:00,2016-04-27 06:00:00+00:00,7,all 7 weather sensors missing,kept NaN (gap > fill limit)
6,plant_zero_daytime,2016-05-25 16:30:00+00:00,2016-05-25 16:45:00+00:00,2,rows with all arrays zero: 2,kept unchanged (candidate anomaly)
7,weather_all_missing,2016-05-25 16:30:00+00:00,2016-05-27 13:00:00+00:00,179,all 7 weather sensors missing,kept NaN (gap > fill limit)


## 7) التقسيم الزمني
الحدود بالـ UTC: train حتى 2021-12-31، وval سنة 2022، وtest سنة 2023.
- ملفات val وtest بتبدأ بـ 14 يوم warm-up (متعلّمة `is_warmup`) للـ lags، ومش للتقييم.
- آخر 48 ساعة قبل بداية أي split متعلّمة `in_purge_zone` عشان أهداف 24–48 ساعة متتسربش للـ split اللي بعده.
- الـ test فيه عطل حساس الإشعاع في أكتوبر–ديسمبر 2023، فنتايجه محتاجة قراءة مع `wx_complete`.

In [9]:
UTC = "UTC"
BOUNDS = {
    "train": (idx[0], pd.Timestamp("2021-12-31 23:45", tz=UTC)),
    "val": (pd.Timestamp("2022-01-01 00:00", tz=UTC), pd.Timestamp("2022-12-31 23:45", tz=UTC)),
    "test": (pd.Timestamp("2023-01-01 00:00", tz=UTC), idx[-1]),
}
split_label = pd.Series("", index=idx)
for name, (s, e) in BOUNDS.items():
    split_label[(idx >= s) & (idx <= e)] = name
assert (split_label != "").all()

in_purge = pd.Series(False, index=idx)
for nxt in ("val", "test"):
    st = BOUNDS[nxt][0]
    in_purge |= (idx >= st - pd.Timedelta(minutes=STEP_MIN * PURGE_ROWS)) & (idx < st)
print({k: int((split_label == k).sum()) for k in BOUNDS}, "| purge rows:", int(in_purge.sum()))

{'train': 210439, 'val': 35040, 'test': 35033} | purge rows: 384


## 8) بناء المجموعات
- `targets`: القدرة الكلية للمحطة (`plant_pu`).
- `features_weather`: الطقس النضيف (مرصود، محتاج lag قبل الاستخدام كـ feature).
- `arrays`: قدرة الـ 14 array في نفس الوقت. **مش features** (نتيجة نفس الساعة)، بس evidence لتحليل الأعطال.
- `data_quality_flags`: كل الـ flags والقيم الأصلية وتقسيمات الـ splits.

In [10]:
flags = pd.DataFrame(index=idx)
flags["solar_elev_deg"] = elev.round(3)
flags["is_night"], flags["is_day"] = is_night, is_day
for rule, m in rule_rows.items():
    flags[f"rule_{rule}"] = m
flags["wdir_wrapped"] = wrapped
flags["irr_offset_applied_wm2"] = offset_row.round(3)
for c, s in orig_vals.items():
    flags[f"orig_{c}"] = s
flags["wx_all_missing_raw"] = all_missing_raw
flags["wx_imputed_count"] = wx_imputed_count
flags["wx_unfilled_any"] = wx_unfilled_any
flags["wx_complete"] = ~wx_unfilled_any
flags["wx_complete_ex_irr"] = ~w[[c for c in wcols if c != "irr_wm2"]].isna().any(axis=1)
flags["cand_plant_zero_daytime"] = cand_zero
flags["array_outage_count"] = arr_out_count
for c in arr_cols:
    flags[f"array_outage_{c}"] = arr_out[c]
flags["split"] = split_label
flags["in_purge_zone"] = in_purge
flags["is_warmup"] = False
flags["scoreable_plant"] = plant.notna()
flags["scoreable_plant_with_weather"] = plant.notna() & ~wx_unfilled_any

groups = {
    "targets": raw[["plant_pu"]],
    "features_weather": w[weather_cols],
    "arrays": raw[arr_cols],
    "data_quality_flags": flags,
}
for g, f in groups.items():
    print(f"{g:20s} {f.shape}")

targets              (280512, 1)
features_weather     (280512, 9)
arrays               (280512, 14)
data_quality_flags   (280512, 40)


## 9) معاملات الـ scaler (train فقط)
`median` و`q25` و`q75` و`iqr` لكل عمود رقمي من الـ train، من غير ما الداتا تتغيّر.
لو الـ IQR صفر (زي `precip_mmh`) بنستخدم الانحراف المعياري كـ fallback، ولو برضو صفر بنستخدم 1.

In [11]:
train_mask = split_label == "train"
scaler_cols = ["plant_pu"] + weather_cols + arr_cols
src = pd.concat([groups["targets"], groups["features_weather"], groups["arrays"]], axis=1)
params = {}
for c in scaler_cols:
    x = src.loc[train_mask, c].dropna()
    q25, med, q75 = (float(v) for v in x.quantile([0.25, 0.5, 0.75]))
    iqr, std = q75 - q25, float(x.std())
    scale, rule = (iqr, "iqr") if iqr > 0 else ((std, "std_fallback") if std > 0 else (1.0, "unit_fallback"))
    params[c] = {"median": med, "q25": q25, "q75": q75, "iqr": iqr, "std": std, "scale": scale,
                 "scale_rule": rule, "n_train_values": int(len(x))}
fit_sha = hashlib.sha256("|".join(idx[train_mask].strftime("%Y-%m-%dT%H:%M").tolist()).encode()).hexdigest()
scaler = {
    "method": "robust: (x - median) / scale",
    "fit_window": {"start_utc": str(idx[train_mask].min()), "end_utc": str(idx[train_mask].max()),
                   "rows": int(train_mask.sum()), "index_sha256": fit_sha},
    "applies_to": "plant_pu, cleaned weather, wind_u/wind_v, arrays. Features created later (lags/rolling) need their own train-only scaler.",
    "not_scaled": "flags, split labels, solar_elev_deg",
    "refit_note": "A final refit on train+val must be saved as a new versioned file with its own fit_window.",
    "params": params,
}
print("columns with IQR == 0:", [c for c, p in params.items() if p["scale_rule"] != "iqr"])

columns with IQR == 0: ['precip_mmh']


## 10) كتابة الملفات
كل ملف بيتكتب كـ `.csv.gz` (ضغط بدون timestamp عشان الـ hash يفضل ثابت).
المفروض تقرأه بـ `pd.read_csv(path, index_col="time_utc", parse_dates=True)`.
الأعمدة المنطقية بتتكتب 0/1 عشان الحجم.

In [12]:
OUTPUTS = []


def to_csv_text(frame):
    f = frame.copy()
    for c in f.columns:
        if f[c].dtype == bool:
            f[c] = f[c].astype("int8")
    return f.to_csv(index_label="time_utc", float_format="%.4f")


def write_csv_gz(frame, name):
    with open(f"{OUT_DIR}/{name}", "wb") as raw_fh:
        with gzip.GzipFile(filename="", mode="wb", fileobj=raw_fh, mtime=0) as gz:
            gz.write(to_csv_text(frame).encode())
    OUTPUTS.append(name)


def make_split(frame, name, is_flags=False):
    s, e = BOUNDS[name]
    start = s if name == "train" else s - pd.Timedelta(minutes=STEP_MIN * WARMUP_ROWS)
    out = frame.loc[start:e].copy()
    if is_flags:
        out["is_warmup"] = out.index < s
        out["scoreable_plant"] = out["scoreable_plant"] & ~out["is_warmup"]
        out["scoreable_plant_with_weather"] = out["scoreable_plant_with_weather"] & ~out["is_warmup"]
    return out


for g, frame in groups.items():
    for name in BOUNDS:
        write_csv_gz(make_split(frame, name, is_flags=(g == "data_quality_flags")), f"{name}_{g}.csv.gz")

events_df.to_csv(f"{OUT_DIR}/events.csv", index=False); OUTPUTS.append("events.csv")
print(len(OUTPUTS), "files written")

13 files written


In [13]:
month_cov = {k: pd.Series(idx[split_label == k].tz_convert(LOCAL_TZ).month).value_counts().sort_index().to_dict() for k in BOUNDS}
quarters = pd.date_range("2021-01-01", "2022-10-01", freq="QS", tz=UTC)
cv_folds = []
for i, q in enumerate(quarters, 1):
    v_end = (q + pd.offsets.QuarterEnd(0)).normalize() + pd.Timedelta(hours=23, minutes=45)
    cv_folds.append({"fold": i, "train_first_origin": str(idx[0]),
                     "train_last_origin": str(q - pd.Timedelta(minutes=STEP_MIN * (PURGE_ROWS + 1))),
                     "val_start": str(q), "val_end": str(v_end)})
known_issues = {k: [] for k in BOUNDS}
for _, ev_ in events_df[events_df["type"].isin(["irradiance_sensor_fault", "irradiance_offset", "temp_sensor_episode",
                                                  "module_temp_stuck", "weather_all_missing"])].iterrows():
    if ev_["type"] == "weather_all_missing" and ev_["rows"] < 96:
        continue
    for k, (bs, be) in BOUNDS.items():
        a, b = max(ev_["start_utc"], bs), min(ev_["end_utc"], be)
        if a <= b:
            known_issues[k].append(f"{ev_['type']}: {a.date()} .. {b.date()}")
splits = {
    "timezone": "UTC", "step_minutes": STEP_MIN,
    "bounds": {k: {"start": str(s), "end": str(e), "rows": int((split_label == k).sum())} for k, (s, e) in BOUNDS.items()},
    "warmup_rows": WARMUP_ROWS, "purge_rows": PURGE_ROWS,
    "rows_per_day": 96, "horizon_rows": {"24h": 96, "48h": 192},
    "rows_in_files_including_warmup": {k: int(len(make_split(groups["targets"], k))) for k in BOUNDS},
    "how_to_use": ("val_*/test_* files start with warm-up rows (is_warmup=1): use them only to build lags/rolling features, "
                   "never for training or scoring. Drop forecast origins flagged in_purge_zone when building training samples."),
    "months_coverage_rows_athens_time": month_cov,
    "known_data_issues_by_split": known_issues,
    "optional_rolling_origin_cv": cv_folds,
}
dump_json(splits, f"{OUT_DIR}/splits.json"); OUTPUTS.append("splits.json")
dump_json(scaler, f"{OUT_DIR}/scaler_params.json"); OUTPUTS.append("scaler_params.json")

column_groups = {
    "targets": {"columns": ["plant_pu"], "role": "label: normalised plant active power"},
    "features_weather": {"columns": weather_cols, "role": "OBSERVED weather (same time step)"},
    "arrays": {"columns": arr_cols, "role": "same-step outcomes per array: anomaly evidence only, NOT features"},
    "data_quality_flags": {"columns": list(flags.columns), "role": "flags, masks, original values, split labels"},
}
availability = {
    "forecast_protocol_ASSUMPTION": {
        "note": "Default protocol proposed during preprocessing; change it deliberately before feature engineering.",
        "forecast_origin": "every day at 00:00 UTC",
        "horizons_rows": "1..192 after the origin (96 = 24h, 192 = 48h; report both)",
        "known_at_origin": ["plant and array history up to the origin", "observed weather up to the origin", "deterministic calendar and solar geometry"],
        "not_known_at_origin": ["any weather value inside the forecast window (the dataset has no weather forecasts)"],
    },
    "columns": {
        "targets": "history only: lags >= horizon",
        "features_weather": "observed, not forecast: lag by >= horizon, or use declared weather scenarios",
        "arrays": "same-step outcomes: forbidden as forecasting features (lags only)",
        "data_quality_flags": "never a model feature; use for masking/evaluation only (is_night/solar_elev_deg are deterministic and may be recomputed as features)",
    },
}
dump_json(column_groups, f"{OUT_DIR}/column_groups.json"); OUTPUTS.append("column_groups.json")
dump_json(availability, f"{OUT_DIR}/feature_availability.json"); OUTPUTS.append("feature_availability.json")
print(len(OUTPUTS), "artifacts written")

17 artifacts written


## 11) التقرير النهائي وFinal Validation
التقرير فيه SHA256 لكل الملفات (ما عدا التقرير نفسه) وللملفات الأصلية.
الـ Final Validation بيعيد قراءة الملفات من الديسك ويتأكد من كل الشروط.

In [14]:
REPORT["site_assumption"] = {"lat": SITE_LAT, "lon": SITE_LON,
                             "note": "approximate; longitude inferred from solar-noon timing, latitude has little effect on the night mask (38-42N tested)"}
REPORT["rule_hits"] = rule_hits
REPORT["temp_sensor_episode"] = {"start_utc": str(ep_start), "end_utc_exclusive": str(ep_end)}
REPORT["irradiance_fault_windows_utc_days"] = fault_windows
REPORT["irradiance_offset"] = {"rows_corrected": int(off_rows.sum()), "first_utc": str(off_rows[off_rows].index.min()),
                               "last_utc": str(off_rows[off_rows].index.max()), "method": "causal: median of the previous 7 daily night medians"}
REPORT["weather_missing"] = {"all_missing_rows_raw": int(all_missing_raw.sum()), "imputed_values": int(imputed.sum().sum()),
                             "rows_still_incomplete": int(wx_unfilled_any.sum()), "ffill_limit_rows": FFILL_LIMIT,
                             "irradiance_ffill_limit_rows": IRR_FFILL_LIMIT}
REPORT["events_summary"] = events_df["type"].value_counts().to_dict()
REPORT["candidates"] = {"plant_zero_daytime_rows": int(cand_zero.sum()),
                        "naive_rule_rows_for_comparison": int(naive_rule.sum()),
                        "array_outage_rows": {c: int(arr_out[c].sum()) for c in arr_cols}}
REPORT["time_split"] = {k: splits[k] for k in ("timezone", "bounds", "warmup_rows", "purge_rows")}
REPORT["limitations"] = [
    "Site latitude/longitude are approximations; only used for the day/night masks.",
    "Plant-zero and array-outage candidates are proxies (curtailment or maintenance can also cause zeros), not verified labels.",
    "Observed weather is not a forecast; using it without lags overstates 24-48h accuracy.",
    "The irradiance offset estimate is causal but lags 3-4 days at its onset and is not applied inside fault windows (irradiance is NaN there).",
    "Thresholds were chosen after inspecting the full dataset, including the 2023 test year.",
    "Test year 2023 has no valid irradiance from 2023-10-12 onwards.",
]
REPORT["output_sha256"] = {n: sha256_file(f"{OUT_DIR}/{n}") for n in sorted(OUTPUTS)}
REPORT["output_sha256_note"] = "data_quality_report.json is excluded because it contains these hashes."
dump_json(REPORT, f"{OUT_DIR}/data_quality_report.json"); OUTPUTS.append("data_quality_report.json")
print("report written")

report written


In [15]:
# ================= Final Validation =================
rd = lambda n: pd.read_csv(f"{OUT_DIR}/{n}", index_col="time_utc", parse_dates=True)
rep = json.load(open(f"{OUT_DIR}/data_quality_report.json"))

# 1) الملفات بتطابق الـ hashes المسجلة، وتحقق الـ timestamps المسجّل
for n, h in rep["output_sha256"].items():
    assert sha256_file(f"{OUT_DIR}/{n}") == h, f"hash mismatch: {n}"
tr_ = rep["time_reconstruction"]
assert tr_["local_match_ratio"] == 1.0 and tr_["spring_forward_jumps"] == tr_["fall_back_repeats"]

# 2) splits زمنية متصلة وغير متداخلة (بعد استبعاد الـ warm-up) وبتغطي الداتا بالكامل
GROUPS = list(groups)
core = {}
for name in BOUNDS:
    f = rd(f"{name}_data_quality_flags.csv.gz")
    c = f.index[f["is_warmup"] == 0]
    assert c.is_unique and c.is_monotonic_increasing
    assert (c.to_series().diff().dropna() == pd.Timedelta(minutes=STEP_MIN)).all(), f"gap inside {name}"
    assert (f.loc[c, "split"] == name).all()
    core[name] = (c.min(), c.max(), len(c))
    if name != "train":
        assert int(f["is_warmup"].sum()) == WARMUP_ROWS and f.index[f["is_warmup"] == 1].max() < c.min()
assert core["val"][0] - core["train"][1] == pd.Timedelta(minutes=STEP_MIN)
assert core["test"][0] - core["val"][1] == pd.Timedelta(minutes=STEP_MIN)
assert sum(v[2] for v in core.values()) == len(idx), "splits do not cover the dataset exactly"

# 3) نفس الـ index وعدد الصفوف في كل مجموعات الملفات
for name in BOUNDS:
    ref = rd(f"{name}_targets.csv.gz").index
    for g in GROUPS:
        cur = rd(f"{name}_{g}.csv.gz").index
        assert len(cur) == len(ref) and (cur == ref).all(), f"index mismatch in {name}_{g}"

# 4) المدى الفيزيائي (آمن مع NaN) من الملفات المكتوبة
def rng(frame, col, lo, hi):
    x = frame[col].dropna()
    assert len(x) == 0 or (x.min() >= lo and x.max() <= hi), f"{col} out of range: {x.min()}..{x.max()}"

for name in BOUNDS:
    tg, wx, ar, fl = (rd(f"{name}_{g}.csv.gz") for g in GROUPS)
    rng(tg, "plant_pu", 0, 1)
    for c in arr_cols: rng(ar, c, 0, 1)
    rng(wx, "temp_c", -30, 50); rng(wx, "irr_wm2", 0, IRR_MAX); rng(wx, "mod_temp_c", -30, 90)
    rng(wx, "rh_pct", 0, 100); rng(wx, "wspd_ms", 0, WSPD_MAX); rng(wx, "wdir_deg", 0, 360); rng(wx, "precip_mmh", 0, 100)
    # أي NaN في الطقس لازم يكون متعلّم، والـ flag متسق مع الأعمدة
    assert ((wx[wcols].isna().any(axis=1).astype(int)) == fl["wx_unfilled_any"]).all()
    # كل صفوف عطل الإشعاع لازم الإشعاع فيها NaN
    assert wx.loc[fl["rule_irr_fault_window"] == 1, "irr_wm2"].isna().all()

# 5) الـ scaler اتحسب من train بس
sp = json.load(open(f"{OUT_DIR}/scaler_params.json"))
assert pd.Timestamp(sp["fit_window"]["end_utc"]) <= core["train"][1]
assert all(p["scale"] > 0 for p in sp["params"].values())

# 6) الـ index نفسه منتظم (15 دقيقة) والـ test فيه آخر صف متوقع
assert core["test"][1] == last_expected
print("FINAL VALIDATION PASSED")
print("splits (core rows):", {k: v[2] for k, v in core.items()})

FINAL VALIDATION PASSED
splits (core rows): {'train': 210439, 'val': 35040, 'test': 35033}


In [16]:
print(f"{'file':42s} {'size':>12s}")
for n in sorted(OUTPUTS):
    print(f"{n:42s} {os.path.getsize(f'{OUT_DIR}/{n}'):>12,d}")

file                                               size
column_groups.json                                1,840
data_quality_report.json                          8,442
events.csv                                       41,158
feature_availability.json                         1,007
scaler_params.json                                6,509
splits.json                                       4,437
test_arrays.csv.gz                              719,581
test_data_quality_flags.csv.gz                  268,437
test_features_weather.csv.gz                  1,130,398
test_targets.csv.gz                             144,428
train_arrays.csv.gz                           4,170,990
train_data_quality_flags.csv.gz               1,309,852
train_features_weather.csv.gz                 6,591,841
train_targets.csv.gz                            833,809
val_arrays.csv.gz                               693,750
val_data_quality_flags.csv.gz                   255,749
val_features_weather.csv.gz                   1,